### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="bank_customer_churn",
    dataset_year="2020",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/gauravtopre/bank-customer-churn-dataset",
    download_description="""
We download the data from Kaggle to a .csv file in a predefined folder.
    
mkdir -p local-data-warehouse/bank_customer_churn && cd local-data-warehouse/bank_customer_churn && kaggle datasets download gauravtopre/bank-customer-churn-dataset && unzip bank-customer-churn-dataset.zip && rm bank-customer-churn-dataset.zip && cd ../../
""",
    # References
    academic_reference_bibtex="""@misc{Topre2022BankCustomerChurn,
  title={Bank Customer Churn Dataset},
  author={Gaurav Topre},
  year={2022},
  publisher={Kaggle},
  url={https://www.kaggle.com/datasets/gauravtopre/bank-customer-churn-dataset}
}
""",
    academic_reference_bibtex_key="Topre2022BankCustomerChurn",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We remove the customer_id column.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="churn",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="churn",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "Bank Customer Churn Prediction.csv")

df["churn"] = df["churn"].map({1: "Yes", 0: "No"})

cat_cols = ["churn", "active_member", "country", "gender", "credit_card"]
df[cat_cols] = df[cat_cols].astype("category")

df = df.drop(columns=["customer_id"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 10,000
Columns: 11
Use sampling: False (sample size: 10,000)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['estimated_salary', 'balance', 'credit_score', 'age', 'tenure', 'products_number', 'country', 'credit_card', 'gender', 'active_member']
Rows remaining as candidates after top-10 filter: 0 (of 10,000)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,596,Germany,Male,32,3,96709.07,2,0,0,41788.37,No
1,623,France,Male,43,1,0.00,2,1,1,146379.30,No
2,601,Spain,Female,44,4,0.00,2,1,0,58561.31,No
3,506,Germany,Male,59,8,119152.10,2,1,1,170679.74,No
4,560,Spain,Female,27,7,124995.98,1,1,1,114669.79,No


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,country,category,0.0,0.0,3.0,"France, Germany, Spain"
1,gender,category,0.0,0.0,2.0,"Male, Female"
2,credit_card,category,0.0,0.0,2.0,"1, 0"
3,active_member,category,0.0,0.0,2.0,"1, 0"
4,churn,category,0.0,0.0,2.0,"No, Yes"
5,balance,float64,0.0,0.0,6382.0,"0.0, 105473.74, 130170.82, 145018.64, 136855.94, 135795.63, 112713.34, 88963.31, 119787.76, 121535.18"
6,estimated_salary,float64,0.0,0.0,9999.0,"24924.92, 143301.49, 10023.15, 87609.5, 9903.42, 64831.36, 40313.47, 164248.33, 57876.05, 172665.21"
7,credit_score,int64,0.0,0.0,460.0,"850, 678, 655, 705, 667, 684, 670, 651, 648, 683"
8,age,int64,0.0,0.0,70.0,"37, 38, 35, 36, 34, 33, 40, 39, 32, 31"
9,tenure,int64,0.0,0.0,11.0,"2, 1, 7, 8, 5, 3, 4, 9, 6, 10"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
credit_score,10000.0,650.528800,96.653299,350.00,850.00
age,10000.0,38.921800,10.487806,18.00,92.00
tenure,10000.0,5.012800,2.892174,0.00,10.00
balance,10000.0,76485.889288,62397.405202,0.00,250898.09
products_number,10000.0,1.530200,0.581654,1.00,4.00
estimated_salary,10000.0,100090.239881,57510.492818,11.58,199992.48


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column        rank                       
active_member 1           1   5151  51.51
              2           0   4849  48.49
churn         1          No   7963  79.63
              2         Yes   2037  20.37
country       1      France   5014  50.14
              2     Germany   2509  25.09
              3       Spain   2477  24.77
credit_card   1           1   7055  70.55
              2           0   2945  29.45
gender        1        Male   5457  54.57
              2      Female   4543  45.43

In [8]:
# Target Distribution
target_df

,count,pct
churn,,
No,7963,79.63
Yes,2037,20.37


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to bank_customer_churn/019d7366-99cc-77cc-8d41-5c3ccc8eff96
019d7366-99cc-77cc-8d41-5c3ccc8eff96
631a7b4311ef96cce0271f48f074a4fe344c661f839a30eb632e990647912c5a
